In [ ]:
import cv2
import torch
import numpy as np
from collections import deque
from ultralytics import YOLO
from catboost import CatBoostRegressor

In [ ]:
dataset_rows = []
model = YOLO(r'C:\Users\user\projectolymp\runs\detect\models\110\weights\best.pt')
cap = cv2.VideoCapture(0)
track_history = {}
cnt = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: 
        break
    results = model.track(frame, persist=True, tracker="bytetrack.yaml") # ищет маркеры и присваевает им id
    if results[0].boxes.id is not None: # провеййрка нашла ли модель хоть 1 маркер
        boxes = results[0].boxes.xywh.cpu().numpy() # получение координатов объектов
        track_ids = results[0].boxes.id.int().cpu().tolist() # получение id
        for box, track_id in zip(boxes, track_ids): # проходим по координате и ее id
            x, y, w, h = box # распаковка
            track = track_history.get(track_id, deque(maxlen=150)) # история 5 секунд при 30 фпс
            track.append((float(x), float(y))) # добавляем точку в историю движения 
            track_history[track_id] = track # сохраняем обновленную историю
            if len(track) > 25: 
                past = list(track)[-25:-15] # 10 точек прошлого
                future = list(track)[-1]    # точка через 15 кадров после начала прошлого
                row = [c for pt in past for c in pt] + [future[0], future[1]]
                dataset_rows.append(row)
            for i in range(1, len(track)): # отрисовка
                pt1 = (int(track[i-1][0]), int(track[i-1][1]))
                pt2 = (int(track[i][0]), int(track[i][1]))
                cv2.line(frame, pt1, pt2, (0, 0, 255), 2) # красная линия толщиной 2 пикселя
        cnt += 1
    cv2.imshow("MMC", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): # на q закрытие
        break
cap.release()
cv2.destroyAllWindows()

data_array = np.array(dataset_rows) # превращение в массив numpy
X = data_array[:, :-2] # признаки
Y = data_array[:, -2:] # таргет
cb_model = CatBoostRegressor(iterations=1000, 
    loss_function='MultiRMSE', 
    depth=6,    
    learning_rate=0.1,
    verbose=100)
cb_model.fit(X, Y)
cb_model.save_model("catboost.cbm")